In [1]:
import zipfile
from pathlib import Path

zip_path = Path(r"C:\Users\harish\Downloads\archive (1).zip")
extract_to = Path(r"C:\Users\harish\Downloads\sub project")

extract_to.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)

print("Extraction complete.")

Extraction complete.


In [2]:
from pathlib import Path

ROOT = Path(r"C:\Users\harish\Downloads\sub project\FracAtlas")
IMAGE_DIR = ROOT / "images" / "Fractured"
COCO_JSON = ROOT / "Annotations" / "COCO JSON" / "COCO_fracture_masks.json"

print("Image folder exists:", IMAGE_DIR.exists())
print("COCO JSON exists:", COCO_JSON.exists())

Image folder exists: True
COCO JSON exists: True


In [3]:
import os

YOLO_ROOT = ROOT / "yolo_dataset"

for split in ["train", "val"]:
    (YOLO_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

print("YOLO folders created")

YOLO folders created


In [4]:
import json
import shutil
import random
from tqdm import tqdm

with open(COCO_JSON) as f:
    coco = json.load(f)

images_info = coco["images"]
annotations = coco["annotations"]

# Create mapping image_id → annotations
ann_dict = {}
for ann in annotations:
    img_id = ann["image_id"]
    if img_id not in ann_dict:
        ann_dict[img_id] = []
    ann_dict[img_id].append(ann)

# Shuffle images
random.shuffle(images_info)
split_idx = int(0.8 * len(images_info))
train_images = images_info[:split_idx]
val_images = images_info[split_idx:]

def process_split(image_list, split_name):
    for img_info in tqdm(image_list):
        file_name = img_info["file_name"]
        img_id = img_info["id"]
        width = img_info["width"]
        height = img_info["height"]

        img_path = IMAGE_DIR / file_name
        if not img_path.exists():
            continue

        shutil.copy(img_path, YOLO_ROOT / "images" / split_name)

        label_path = YOLO_ROOT / "labels" / split_name / (Path(file_name).stem + ".txt")

        with open(label_path, "w") as f:
            if img_id in ann_dict:
                for ann in ann_dict[img_id]:
                    segmentation = ann["segmentation"]
                    for seg in segmentation:
                        seg = [seg[i] / width if i % 2 == 0 else seg[i] / height for i in range(len(seg))]
                        seg_str = " ".join(map(str, seg))
                        f.write(f"0 {seg_str}\n")

process_split(train_images, "train")
process_split(val_images, "val")

print("COCO → YOLO conversion complete")

100%|██████████| 144/144 [00:02<00:00, 51.48it/s]

COCO → YOLO conversion complete


In [5]:
import yaml

data_yaml = {
    "path": str(YOLO_ROOT),
    "train": "images/train",
    "val": "images/val",
    "names": {0: "fracture"}
}

with open(YOLO_ROOT / "data.yaml", "w") as f:
    yaml.dump(data_yaml, f)

print("data.yaml created")

data.yaml created


In [1]:
from pathlib import Path
import torch
from ultralytics import YOLO

# Define necessary paths
ROOT = Path(r"C:\Users\harish\Downloads\sub project\FracAtlas")
YOLO_ROOT = ROOT / "yolo_dataset"

device = 0 if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model = YOLO("yolov8m-seg.pt")

model.train(
    data=str(YOLO_ROOT / "data.yaml"),
    epochs=30,
    imgsz=416,
    batch=4,
    device=device,
    lr0=1e-3,
    optimizer="AdamW",
    patience=10
)

Using device: cpu
Ultralytics 8.4.14  Python-3.14.2 torch-2.10.0+cpu CPU (Intel Core(TM) i5-10300H 2.50GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\harish\Downloads\sub project\FracAtlas\yolo_dataset\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train8, nbs=64, nms=False, opset=None, optimi

ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001C263D7CAD0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.0410

In [1]:
import pandas as pd

df = pd.read_csv("runs/segment/train8/results.csv")

# Show last epoch results
print(df.tail(1))

    epoch     time  train/box_loss  train/seg_loss  train/cls_loss  \
29     30  30195.1         1.77626         2.59156         1.22188   

    train/dfl_loss  train/sem_loss  metrics/precision(B)  metrics/recall(B)  \
29         1.24381               0               0.82562            0.70404   

    metrics/mAP50(B)  ...  metrics/mAP50(M)  metrics/mAP50-95(M)  \
29           0.77545  ...           0.74797              0.32669   

    val/box_loss  val/seg_loss  val/cls_loss  val/dfl_loss  val/sem_loss  \
29       1.58058       2.48205       1.06016       1.14938             0   

      lr/pg0    lr/pg1    lr/pg2  
29  0.000043  0.000043  0.000043  

[1 rows x 23 columns]
